Objetivo: realizar web scrapping para obtener dos libros

Librerias:
- pip install beautifulsoup4 fpdf weasyprint

In [14]:
from bs4 import BeautifulSoup
import requests
from fpdf import FPDF

In [ ]:

base_url = "https://basecamp.com/gettingreal"

# 1. Obtener lista de capítulos
response = requests.get(base_url)
soup = BeautifulSoup(response.text, "html.parser")


In [6]:
# Encontrar todos los enlaces a capítulos
chapter_links = []
for a in soup.select("a[href]"):
    href = a["href"]
    if href.startswith("/gettingreal/") and href != "/gettingreal/":
        chapter_links.append("https://basecamp.com" + href)


In [ ]:
# 2. preparar el PDF
pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.set_font("Arial", size=12)

# 3. Recorrer capítulos y añadir al PDF
for link in chapter_links:
    res = requests.get(link)
    chapter_soup = BeautifulSoup(res.text, "html.parser")

    title = chapter_soup.find("h1").get_text(strip=True)
    content_elem = chapter_soup.select_one("main") or chapter_soup.select_one(".chapter")
    chapter_text = content_elem.get_text("\n", strip=True) if content_elem else ""

    pdf.add_page()
    pdf.set_font("Arial", "B", 16)
    pdf.multi_cell(0, 10, title)
    pdf.ln()

    pdf.set_font("Arial", size=12)
    pdf.multi_cell(0, 8, chapter_text)

# 4. Guardar el PDF
pdf.output("getting_real.pdf")
print("Libro guardado en getting_real.pdf")

RuntimeError: TTF Font file not found: DejaVuSans.ttf

In [16]:
import requests
from bs4 import BeautifulSoup
from weasyprint import HTML

base_url = "https://basecamp.com/gettingreal"

# 1. Obtener lista de capítulos
response = requests.get(base_url)
soup = BeautifulSoup(response.text, "html.parser")

chapter_links = []
for a in soup.select("a[href]"):
    href = a["href"]
    if href.startswith("/gettingreal/") and href != "/gettingreal/":
        chapter_links.append("https://basecamp.com" + href)

chapter_links = sorted(set(chapter_links))

# 2. Construir HTML del libro
html_content = "<h1>Getting Real - Basecamp</h1>"
for link in chapter_links:
    res = requests.get(link)
    chapter_soup = BeautifulSoup(res.text, "html.parser")

    title = chapter_soup.find("h1").get_text(strip=True)
    content_elem = chapter_soup.select_one("main") or chapter_soup.select_one(".chapter")
    chapter_text = content_elem.prettify() if content_elem else ""

    html_content += f"<h2>{title}</h2>{chapter_text}<hr>"

# 3. Generar PDF
HTML(string=html_content).write_pdf("getting_real.pdf")

print("Libro guardado en getting_real.pdf")


Libro guardado en getting_real.pdf


In [18]:
import requests
from bs4 import BeautifulSoup
from weasyprint import HTML

base_url = "https://basecamp.com/gettingreal"

# 1. Obtener lista de capítulos
response = requests.get(base_url)
response.encoding = "utf-8"  # Fuerza UTF-8
soup = BeautifulSoup(response.text, "html.parser")

chapter_links = []
for a in soup.select("a[href]"):
    href = a["href"]
    if href.startswith("/gettingreal/") and href != "/gettingreal/":
        chapter_links.append("https://basecamp.com" + href)

chapter_links = sorted(set(chapter_links))

# 2. Construir HTML del libro
html_content = """
<h1 style='text-align:center;'>Getting Real - Basecamp</h1>
<p style='text-align:center; font-size:14px;'>Libro recopilado automáticamente para uso personal</p>
<hr>
"""

for i, link in enumerate(chapter_links):
    res = requests.get(link)
    res.encoding = "utf-8"  # Asegura UTF-8
    chapter_soup = BeautifulSoup(res.text, "html.parser")

    # Título del capítulo
    title = chapter_soup.find("h1").get_text(strip=True)

    # Contenido principal
    content_elem = chapter_soup.select_one("main") or chapter_soup.select_one(".chapter")
    if not content_elem:
        continue

    # 3. Eliminar botones "Next Chapter" y textos finales no deseados
    for unwanted in content_elem.find_all(["a", "button"], recursive=True):
        unwanted.decompose()

    # Eliminar texto publicitario final
    for p in content_elem.find_all("p"):
        if "We made Basecamp" in p.get_text() or "Copyright" in p.get_text():
            p.decompose()

    # Convertir contenido limpio a HTML
    chapter_html = content_elem.prettify()

    # Evitar repetir índice/título principal en capítulos después del primero
    html_content += f"<h2>{title}</h2>{chapter_html}<hr>"

# 4. Generar PDF
HTML(string=html_content).write_pdf("getting_real_v2.pdf")
print("Libro guardado en getting_real.pdf")


Libro guardado en getting_real.pdf
